In [ ]:
# Installs and check nvidia-smi is working

!pip install -q transformers torch accelerate pynvml datasets pandas
!nvidia-smi

In [ ]:
# Load benchmark prompts
# 100 MMLU-Pro + 100 XSum + 100 MBPP+ prompts

!pip install -q datasets pandas evalplus

from datasets import load_dataset
from evalplus.data import get_mbpp_plus
import pandas as pd
import random


SEED = 42
N_PER_TASK = 100

rows = []


# ================================================================
# MMLU-Pro: factual question-answering
# ================================================================

mmlu = load_dataset(
    "TIGER-Lab/MMLU-Pro",
    split="test"
)

mmlu = (
    mmlu
    .shuffle(seed=SEED)
    .select(range(N_PER_TASK))
)

for ex in mmlu:
    options_text = "\n".join(
        f"{chr(65 + i)}. {option.strip()}"
        for i, option in enumerate(ex["options"])
    )

    prompt = f"""Answer the following multiple-choice question.

Your entire response must be exactly one capital letter from A through J.
Do not provide an explanation or any other text.

Question:
{ex["question"]}

Options:
{options_text}
"""

    rows.append({
        "prompt_id": f'mmlu_{ex["question_id"]}',
        "task": "factual_qa",
        "prompt": prompt,
        "reference": ex["answer"]
    })


# ================================================================
# XSum: summarization
# ================================================================

xsum = load_dataset(
    "EdinburghNLP/xsum",
    split="test"
)

# Keep article lengths reasonably controlled.
xsum = xsum.filter(
    lambda x: 250 <= len(x["document"].split()) <= 600
)

xsum = (
    xsum
    .shuffle(seed=SEED)
    .select(range(N_PER_TASK))
)

for ex in xsum:
    prompt = f"""Summarize the following article in exactly one sentence.
Preserve the most important factual information and do not add information
that is not present in the article.

Article:
{ex["document"]}
"""

    rows.append({
        "prompt_id": f'xsum_{ex["id"]}',
        "task": "summarization",
        "prompt": prompt,
        "reference": ex["summary"]
    })


# ================================================================
# MBPP+: coding
# ================================================================

mbpp = get_mbpp_plus()

# Sort before sampling so seed-based selection is reproducible.
task_ids = sorted(mbpp.keys())
selected_ids = random.Random(SEED).sample(
    task_ids,
    N_PER_TASK
)

for task_id in selected_ids:
    problem = mbpp[task_id]

    prompt = f"""Write the complete Python solution for the following task.

Return only valid Python code.
Include the complete function definition.
Do not use Markdown code fences and do not include an explanation.

{problem["prompt"]}
"""

    rows.append({
        "prompt_id": task_id,
        "task": "coding",
        "prompt": prompt,
        "reference": task_id
    })


# ================================================================
# Create and verify benchmark dataframe
# ================================================================

benchmark_df = pd.DataFrame(rows)

expected_counts = {
    "factual_qa": N_PER_TASK,
    "summarization": N_PER_TASK,
    "coding": N_PER_TASK,
}

actual_counts = benchmark_df["task"].value_counts().to_dict()

assert len(benchmark_df) == 3 * N_PER_TASK, (
    f"Expected {3 * N_PER_TASK} prompts, got {len(benchmark_df)}"
)

assert actual_counts == expected_counts, (
    f"Unexpected task counts: {actual_counts}"
)

benchmark_df.to_csv(
    "benchmark_prompts.csv",
    index=False
)

print(f"Collected {len(benchmark_df)} total prompts")
print("\nPrompts by task:")
print(benchmark_df["task"].value_counts())

display(
    benchmark_df[
        ["prompt_id", "task", "prompt", "reference"]
    ].head(10)
)


In [ ]:
# Review benchmark prompt categories

print("Prompt counts by task:")
display(
    benchmark_df.groupby("task")
    .size()
    .rename("count")
)

print("\nSample prompt from each task:")
for task in benchmark_df["task"].unique():
    print(f"\n--- {task} ---")
    print(
        benchmark_df[
            benchmark_df["task"] == task
        ].iloc[0]["prompt"][:1200]
    )


In [ ]:
# Validate the mass-data benchmark before inference

EXPECTED_PROMPTS = 300

assert len(benchmark_df) == EXPECTED_PROMPTS, (
    f"Expected {EXPECTED_PROMPTS} prompts, got {len(benchmark_df)}"
)

assert (
    benchmark_df["task"].value_counts().to_dict()
    == {
        "factual_qa": 100,
        "summarization": 100,
        "coding": 100,
    }
), "Benchmark task counts are not 100/100/100."

assert benchmark_df["prompt_id"].is_unique, (
    "Duplicate prompt_id values found."
)

print("Benchmark validation passed.")
print(f"Total prompts: {len(benchmark_df)}")
print(benchmark_df["task"].value_counts())


In [ ]:
# Sample GPU Baseline Power

!pip install -q nvidia-ml-py

import pynvml

POWER_AVAILABLE = False
gpu_handle = None

try:
    pynvml.nvmlInit()
    gpu = pynvml.nvmlDeviceGetHandleByIndex(0)

    # Variables used by measure_query()
    gpu_handle = gpu
    POWER_AVAILABLE = True

    print("GPU:", pynvml.nvmlDeviceGetName(gpu))
    print(
        "Power:",
        pynvml.nvmlDeviceGetPowerUsage(gpu) / 1000,
        "W"
    )
    print(
        "Memory:",
        pynvml.nvmlDeviceGetMemoryInfo(gpu).total / 1024**3,
        "GB"
    )

except Exception as e:
    print("NVIDIA power measurement unavailable:", e)


In [ ]:
# Download models and perform inference with task-specific token limits
# Expected total: 300 prompts x 2 models = 600 inference runs

import time
import threading
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM


MODELS = {
    "small_1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "large_3B": "Qwen/Qwen2.5-3B-Instruct",
}

TASK_MAX_TOKENS = {
    "factual_qa": 4,
    "summarization": 128,
    "coding": 256,
}

POWER_SAMPLE_INTERVAL = 0.05  # seconds

# Keep prompt metadata attached to every inference.
QUERIES = benchmark_df[
    ["prompt_id", "task", "prompt"]
].to_dict("records")


def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


# Measure latency and GPU-board energy.
def measure_query(
    model,
    tokenizer,
    query,
    max_new_tokens=128
):
    messages = [
        {
            "role": "system",
            "content": "Answer the user's question clearly and concisely."
        },
        {
            "role": "user",
            "content": query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    power_samples = []
    stop_sampling = threading.Event()

    def sample_power():
        while not stop_sampling.is_set():
            try:
                watts = (
                    pynvml.nvmlDeviceGetPowerUsage(
                        gpu_handle
                    )
                    / 1000.0
                )
                power_samples.append(
                    (time.perf_counter(), watts)
                )
            except Exception:
                break

            stop_sampling.wait(
                POWER_SAMPLE_INTERVAL
            )

    sampler = None

    if POWER_AVAILABLE:
        sampler = threading.Thread(
            target=sample_power,
            daemon=True
        )
        sampler.start()

    synchronize_gpu()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    synchronize_gpu()
    elapsed = time.perf_counter() - start

    if sampler is not None:
        stop_sampling.set()
        sampler.join(timeout=2)

    generated_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Integrate sampled power over time.
    energy_j = None
    avg_power_w = None

    if len(power_samples) >= 2:
        energy_j = 0.0

        for j in range(1, len(power_samples)):
            t0, p0 = power_samples[j - 1]
            t1, p1 = power_samples[j]

            energy_j += (
                (p0 + p1) / 2
                * (t1 - t0)
            )

        measured_duration = (
            power_samples[-1][0]
            - power_samples[0][0]
        )

        if measured_duration > 0:
            avg_power_w = (
                energy_j
                / measured_duration
            )
        else:
            energy_j = None

    return {
        "answer": answer,
        "latency_s": elapsed,
        "generated_tokens": len(generated_tokens),
        "tokens_per_s": (
            len(generated_tokens) / elapsed
            if elapsed
            else None
        ),
        "gpu_energy_j": energy_j,
        "avg_gpu_power_w": avg_power_w,
        "power_samples": len(power_samples),
    }


results = []

for model_name, model_id in MODELS.items():
    print(f"\n{'=' * 65}")
    print(f"Loading {model_name}: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id
    )

    dtype = (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    if torch.cuda.is_available():
        model = model.to("cuda")

    model.eval()

    # Warm-up query. Warm-up is not saved as an experiment run.
    _ = measure_query(
        model,
        tokenizer,
        "Hello! Please answer in one sentence.",
        max_new_tokens=16
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for i, item in enumerate(
        QUERIES,
        start=1
    ):
        prompt_id = item["prompt_id"]
        task = item["task"]
        query = item["prompt"]

        token_limit = TASK_MAX_TOKENS[task]

        print(
            f"\n{model_name} | "
            f"Query {i}/{len(QUERIES)} | "
            f"{task} | {prompt_id}"
        )

        measurement = measure_query(
            model,
            tokenizer,
            query,
            max_new_tokens=token_limit
        )

        print(
            f"Answer: "
            f"{measurement['answer'][:300]}"
        )
        print(
            f"Latency: "
            f"{measurement['latency_s']:.3f} s | "
            f"Generated tokens: "
            f"{measurement['generated_tokens']}"
        )

        if measurement["gpu_energy_j"] is not None:
            print(
                f"GPU-board energy: "
                f"{measurement['gpu_energy_j']:.3f} J | "
                f"Avg power: "
                f"{measurement['avg_gpu_power_w']:.2f} W"
            )
        else:
            print(
                "GPU energy unavailable for this measurement."
            )

        results.append({
            "model_name": model_name,
            "model_id": model_id,
            "query_number": i,
            "prompt_id": prompt_id,
            "task": task,
            "query": query,
            "token_limit": token_limit,
            **measurement,
        })

        # Periodic checkpoint in case the runtime disconnects.
        if len(results) % 25 == 0:
            pd.DataFrame(results).to_csv(
                "llm_energy_results_checkpoint.csv",
                index=False
            )

    del model, tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ================================================================
# Save and verify inference results
# ================================================================

results_df = pd.DataFrame(results)

expected_total_runs = (
    len(benchmark_df)
    * len(MODELS)
)

assert len(results_df) == expected_total_runs, (
    f"Expected {expected_total_runs} runs, "
    f"got {len(results_df)}"
)

runs_per_model = (
    results_df["model_name"]
    .value_counts()
    .to_dict()
)

for model_name in MODELS:
    assert runs_per_model.get(model_name, 0) == len(benchmark_df), (
        f"{model_name} has "
        f"{runs_per_model.get(model_name, 0)} runs; "
        f"expected {len(benchmark_df)}."
    )

results_df.to_csv(
    "llm_energy_results.csv",
    index=False
)

print("\nExperiment complete.")
print(
    f"Total inference runs: "
    f"{len(results_df)}"
)
print(
    results_df["model_name"]
    .value_counts()
)

display(
    results_df[
        [
            "model_name",
            "query_number",
            "prompt_id",
            "task",
            "token_limit",
            "latency_s",
            "generated_tokens",
            "tokens_per_s",
            "gpu_energy_j",
            "avg_gpu_power_w"
        ]
    ]
)

print(
    "\nSaved results to "
    "llm_energy_results.csv"
)


In [ ]:
# Repeat MMLU-Pro inference for more stable GPU energy measurements
# 100 prompts x 5 repeats x 2 models = 1,000 energy measurements
#
# Accuracy remains based on the 100 unique MMLU questions.
# These repeats are only for stabilizing the energy estimate.

MMLU_REPEATS = 5

mmlu_queries = (
    benchmark_df[
        benchmark_df["task"] == "factual_qa"
    ][
        ["prompt_id", "prompt"]
    ]
    .to_dict("records")
)

assert len(mmlu_queries) == 100, (
    f"Expected 100 MMLU prompts, got {len(mmlu_queries)}"
)

mmlu_repeat_results = []


for model_name, model_id in MODELS.items():

    print(f"\n{'=' * 65}")
    print(
        f"Loading {model_name} for repeated MMLU energy measurements: "
        f"{model_id}"
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_id
    )

    dtype = (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    if torch.cuda.is_available():
        model = model.to("cuda")

    model.eval()

    # Warm-up is not included in the repeated measurements.
    _ = measure_query(
        model,
        tokenizer,
        "Answer with exactly one capital letter: A",
        max_new_tokens=4
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for prompt_number, item in enumerate(
        mmlu_queries,
        start=1
    ):

        for repeat in range(
            1,
            MMLU_REPEATS + 1
        ):

            measurement = measure_query(
                model,
                tokenizer,
                item["prompt"],
                max_new_tokens=4
            )

            mmlu_repeat_results.append({
                "model_name": model_name,
                "model_id": model_id,
                "prompt_number": prompt_number,
                "prompt_id": item["prompt_id"],
                "repeat": repeat,
                **measurement,
            })

            energy_display = (
                f"{measurement['gpu_energy_j']:.3f} J"
                if measurement["gpu_energy_j"] is not None
                else "NA"
            )

            print(
                f"{model_name} | "
                f"MMLU {prompt_number}/100 | "
                f"repeat {repeat}/{MMLU_REPEATS} | "
                f"{energy_display}"
            )

        if len(mmlu_repeat_results) % 50 == 0:
            pd.DataFrame(
                mmlu_repeat_results
            ).to_csv(
                "mmlu_energy_repeats_checkpoint.csv",
                index=False
            )

    del model, tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ================================================================
# Save and summarize repeated MMLU energy measurements
# ================================================================

mmlu_repeat_df = pd.DataFrame(
    mmlu_repeat_results
)

expected_repeat_measurements = (
    100
    * MMLU_REPEATS
    * len(MODELS)
)

assert len(mmlu_repeat_df) == expected_repeat_measurements, (
    f"Expected {expected_repeat_measurements} repeated MMLU measurements, "
    f"got {len(mmlu_repeat_df)}"
)

mmlu_repeat_df.to_csv(
    "mmlu_energy_repeats.csv",
    index=False
)


# Average the five repeats for each prompt first.
mmlu_per_prompt_energy = (
    mmlu_repeat_df
    .groupby(
        [
            "model_name",
            "prompt_id"
        ],
        as_index=False
    )
    .agg(
        mean_energy_j=(
            "gpu_energy_j",
            "mean"
        ),
        std_energy_j=(
            "gpu_energy_j",
            "std"
        ),
        mean_latency_s=(
            "latency_s",
            "mean"
        ),
        mean_generated_tokens=(
            "generated_tokens",
            "mean"
        ),
        measurements=(
            "gpu_energy_j",
            "count"
        )
    )
)

mmlu_per_prompt_energy.to_csv(
    "mmlu_energy_per_prompt.csv",
    index=False
)


# Then summarize across the 100 unique prompts for each model.
mmlu_energy_summary = (
    mmlu_per_prompt_energy
    .groupby(
        "model_name",
        as_index=False
    )
    .agg(
        average_energy_j=(
            "mean_energy_j",
            "mean"
        ),
        std_across_prompts_j=(
            "mean_energy_j",
            "std"
        ),
        average_latency_s=(
            "mean_latency_s",
            "mean"
        ),
        prompts=(
            "prompt_id",
            "count"
        )
    )
)

mmlu_energy_summary.to_csv(
    "mmlu_energy_summary.csv",
    index=False
)

print(
    "\nRepeated MMLU energy measurement complete."
)

print(
    f"Total repeated measurements: "
    f"{len(mmlu_repeat_df)}"
)

display(
    mmlu_energy_summary
)

print(
    "\nSaved raw repeats to mmlu_energy_repeats.csv"
)

print(
    "Saved per-prompt averages to mmlu_energy_per_prompt.csv"
)

print(
    "Saved model summary to mmlu_energy_summary.csv"
)


In [ ]:
# Grade generated responses
# MMLU-Pro -> exact-match accuracy
# XSum -> ROUGE-L
# MBPP+ -> pass/fail using EvalPlus tests

!pip install -q rouge-score evalplus

import re
import pandas as pd
from rouge_score import rouge_scorer


# ================================================================
# Reload files and refuse to grade stale/mismatched output
# ================================================================

results_df = pd.read_csv(
    "llm_energy_results.csv"
)

benchmark_df = pd.read_csv(
    "benchmark_prompts.csv"
)

benchmark_df = benchmark_df.reset_index(
    drop=True
)
benchmark_df["query_number"] = (
    benchmark_df.index + 1
)

EXPECTED_PROMPTS = 300
EXPECTED_MODELS = 2
EXPECTED_RUNS = (
    EXPECTED_PROMPTS
    * EXPECTED_MODELS
)

assert len(benchmark_df) == EXPECTED_PROMPTS, (
    f"Expected {EXPECTED_PROMPTS} benchmark prompts, "
    f"got {len(benchmark_df)}. "
    "Rerun the benchmark-loading cell."
)

assert len(results_df) == EXPECTED_RUNS, (
    f"Expected {EXPECTED_RUNS} inference runs, "
    f"got {len(results_df)}. "
    "Rerun the mass-data inference cell before grading."
)

for model_name, model_rows in results_df.groupby(
    "model_name"
):
    assert len(model_rows) == EXPECTED_PROMPTS, (
        f"{model_name} has {len(model_rows)} runs; "
        f"expected {EXPECTED_PROMPTS}."
    )

# Check that the prompt IDs/tasks saved during inference still match
# the benchmark row assigned to that query number.
validation_df = results_df.merge(
    benchmark_df[
        ["query_number", "prompt_id", "task"]
    ],
    on="query_number",
    how="left",
    suffixes=("_result", "_benchmark")
)

assert (
    validation_df["prompt_id_result"].astype(str)
    ==
    validation_df["prompt_id_benchmark"].astype(str)
).all(), (
    "Prompt IDs in inference results do not match "
    "benchmark_prompts.csv. Do not grade stale results."
)

assert (
    validation_df["task_result"].astype(str)
    ==
    validation_df["task_benchmark"].astype(str)
).all(), (
    "Task labels in inference results do not match "
    "benchmark_prompts.csv. Do not grade stale results."
)

# Merge references onto the verified inference rows.
graded_df = results_df.merge(
    benchmark_df[
        ["query_number", "reference"]
    ],
    on="query_number",
    how="left"
)

graded_df["prediction"] = None
graded_df["quality_score"] = pd.NA
graded_df["evaluation_metric"] = None


# ================================================================
# 1. MMLU-Pro: exact-match multiple-choice accuracy
# ================================================================

def extract_mc_answer(text):
    text = str(text).strip().upper()

    match = re.search(
        r"\b([A-J])\b",
        text
    )

    if match:
        return match.group(1)

    return None


qa_mask = (
    graded_df["task"]
    == "factual_qa"
)

graded_df.loc[
    qa_mask,
    "prediction"
] = (
    graded_df.loc[
        qa_mask,
        "answer"
    ]
    .apply(extract_mc_answer)
)

graded_df.loc[
    qa_mask,
    "quality_score"
] = (
    graded_df.loc[
        qa_mask,
        "prediction"
    ].astype(str)
    ==
    graded_df.loc[
        qa_mask,
        "reference"
    ].astype(str)
).astype(float)

graded_df.loc[
    qa_mask,
    "evaluation_metric"
] = "accuracy"


# ================================================================
# 2. XSum: ROUGE-L against the human reference summary
# ================================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)


def get_rouge_l(
    generated,
    reference
):
    score = rouge.score(
        str(reference),
        str(generated)
    )

    return score["rougeL"].fmeasure


sum_mask = (
    graded_df["task"]
    == "summarization"
)

graded_df.loc[
    sum_mask,
    "quality_score"
] = (
    graded_df.loc[
        sum_mask
    ].apply(
        lambda row: get_rouge_l(
            row["answer"],
            row["reference"]
        ),
        axis=1
    )
)

graded_df.loc[
    sum_mask,
    "evaluation_metric"
] = "ROUGE-L"



def clean_code(text):
    """Remove only surrounding Markdown code fences from generated code."""
    text = str(text).strip()

    if text.startswith("```python"):
        text = text[len("```python"):].lstrip()
    elif text.startswith("```"):
        text = text[3:].lstrip()

    if text.endswith("```"):
        text = text[:-3].rstrip()

    return text


# ================================================================
# 3. MBPP+: functional correctness on the selected 100 problems
# ================================================================
#
# WARNING:
# This executes model-generated Python.
# Use an isolated runtime/cluster job for untrusted generated code.

from evalplus.data import get_mbpp_plus
from evalplus.evaluate import (
    get_groundtruth,
    check_correctness
)
from evalplus.eval import PASS
from evalplus.eval._special_oracle import (
    MBPP_OUTPUT_NOT_NONE_TASKS
)

coding_mask = (
    graded_df["task"]
    == "coding"
)

coding_rows = graded_df[
    coding_mask
].copy()

if len(coding_rows) > 0:
    all_mbpp = get_mbpp_plus()

    selected_task_ids = set(
        coding_rows[
            "prompt_id"
        ].astype(str)
    )

    selected_problems = {
        task_id: problem
        for task_id, problem
        in all_mbpp.items()
        if str(task_id)
        in selected_task_ids
    }

    assert (
        len(selected_problems)
        == 100
    ), (
        f"Expected 100 selected MBPP+ problems, "
        f"got {len(selected_problems)}."
    )

    # Fresh cache name for the 100-task, seed-42 selection.
    expected_output = get_groundtruth(
        selected_problems,
        "mbpp_plus_100_seed42",
        MBPP_OUTPUT_NOT_NONE_TASKS
    )

    for idx, row in coding_rows.iterrows():
        task_id = str(
            row["prompt_id"]
        )

        if task_id not in selected_problems:
            graded_df.loc[
                idx,
                "prediction"
            ] = "evaluation_error"
            graded_df.loc[
                idx,
                "quality_score"
            ] = 0.0
            graded_df.loc[
                idx,
                "evaluation_metric"
            ] = "MBPP+ pass@1"
            continue

        solution = clean_code(
            row["answer"]
        )

        try:
            result = check_correctness(
                dataset="mbpp",
                completion_id=0,
                problem=selected_problems[
                    task_id
                ],
                solution=solution,
                expected_output=expected_output[
                    task_id
                ],
                base_only=False,
                fast_check=False
            )

            base_status = (
                result["base"][0]
            )
            plus_status = (
                result["plus"][0]
            )

            passed = (
                base_status == PASS
                and plus_status == PASS
            )

            graded_df.loc[
                idx,
                "prediction"
            ] = (
                "pass"
                if passed
                else "fail"
            )

            graded_df.loc[
                idx,
                "quality_score"
            ] = float(passed)

            graded_df.loc[
                idx,
                "evaluation_metric"
            ] = "MBPP+ pass@1"

        except Exception as e:
            graded_df.loc[
                idx,
                "prediction"
            ] = "evaluation_error"

            graded_df.loc[
                idx,
                "quality_score"
            ] = 0.0

            graded_df.loc[
                idx,
                "evaluation_metric"
            ] = "MBPP+ pass@1"

            print(
                f"Evaluation error for "
                f"{row['model_name']} "
                f"{task_id}: {e}"
            )


# ================================================================
# Save and summarize
# ================================================================

graded_df["quality_score"] = pd.to_numeric(
    graded_df["quality_score"],
    errors="coerce"
)

graded_df.to_csv(
    "llm_energy_results_graded.csv",
    index=False
)

print("\nPer-run graded results:")

display(
    graded_df[
        [
            "model_name",
            "query_number",
            "prompt_id",
            "task",
            "evaluation_metric",
            "prediction",
            "quality_score",
            "latency_s",
            "generated_tokens",
            "gpu_energy_j",
            "avg_gpu_power_w"
        ]
    ]
)

print(
    "\nAverage quality, energy, "
    "and latency by model and task:"
)

summary = (
    graded_df
    .groupby(
        [
            "model_name",
            "task",
            "evaluation_metric"
        ],
        dropna=False
    )
    .agg(
        average_quality=(
            "quality_score",
            "mean"
        ),
        average_energy_j=(
            "gpu_energy_j",
            "mean"
        ),
        average_latency_s=(
            "latency_s",
            "mean"
        ),
        average_generated_tokens=(
            "generated_tokens",
            "mean"
        ),
        runs=(
            "query_number",
            "count"
        )
    )
    .reset_index()
)

display(summary)

summary.to_csv(
    "llm_energy_summary.csv",
    index=False
)

print(
    "\nSaved graded results to "
    "llm_energy_results_graded.csv"
)
print(
    "Saved summary to "
    "llm_energy_summary.csv"
)


In [ ]:
# Files generated in the Colab runtime

import os

for filename in [
    "benchmark_prompts.csv",
    "llm_energy_results_checkpoint.csv",
    "llm_energy_results.csv",
    "mmlu_energy_repeats_checkpoint.csv",
    "mmlu_energy_repeats.csv",
    "mmlu_energy_per_prompt.csv",
    "mmlu_energy_summary.csv",
    "llm_energy_results_graded.csv",
    "llm_energy_summary.csv",
]:
    if os.path.exists(filename):
        print(
            filename,
            "->",
            os.path.abspath(filename)
        )
